# 02: LIBEROのタスクを動かす

LIBEROは、ロボットが指示どおりに物を扱えるかをシミュレータで試す、共通の課題集です。タスクごとに、指示や物の配置、成功と判定する条件が変わります。

この回は、LIBERO-Spatialからタスクを1つ選び、SmolVLAを動かして動画を見ます。

[LIBEROの公式リポジトリ](https://github.com/Lifelong-Robot-Learning/LIBERO)

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `02_libero_evaluation` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `common.py`
- `evaluation.json`
- `libero_env.py`
- `prepare.py`
- `requirements.txt`
- `robot_io.py`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/02_libero_evaluation"  # @param {type:"string"}
REQUIRED_FILES = [
    "common.py",
    "evaluation.json",
    "libero_env.py",
    "prepare.py",
    "requirements.txt",
    "robot_io.py",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_02")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## LIBEROでの評価処理

カメラ画像などの観測から行動を予測し、シミュレータを1ステップ進めます。評価を実行するセルでこの処理を使います。

In [ ]:
EVALUATION_CODE = r"""
import hashlib

import imageio.v2 as imageio
import numpy as np

from common import output_directory, write_json
from libero_env import ASSETS_REVISION, open_scene, scene_cases
from robot_io import validate_action


def evaluate(settings, split):
    policy, pre, post, _ = load_model(settings["policy"])
    folder = output_directory("libero_" + split)
    episodes = []

    for task, initial in scene_cases(settings, split):
        seed = settings["seed"] + task * 10000 + initial
        torch.manual_seed(seed)
        np.random.seed(seed)
        policy.reset()
        pre.reset()
        post.reset()
        env, obs, instruction = open_scene(settings, task, initial, seed)
        try:
            state_hash = hashlib.sha256(np.asarray(env.get_sim_state()).tobytes()).hexdigest()
            video_path = folder / f"task_{task:02d}_init_{initial:03d}.mp4"
            success = False
            with imageio.get_writer(video_path, fps=20) as video:
                for step in range(settings["max_steps"]):
                    video.append_data(obs["agentview_image"][::-1, ::-1])
                    action = predict_action(policy, pre, post, obs, instruction, settings["policy"])
                    obs, _, done, _ = env.step(validate_action(action))
                    success = bool(env.check_success())
                    if success or done:
                        break
                video.append_data(obs["agentview_image"][::-1, ::-1])
            episodes.append({
                "task_id": task,
                "initial_state": initial,
                "seed": seed,
                "initial_state_sha256": state_hash,
                "instruction": instruction,
                "steps": step + 1,
                "success": success,
                "video": video_path.name,
            })
        finally:
            env.close()

    successes = sum(episode["success"] for episode in episodes)
    report = {
        "split": split, "settings": settings, "assets_revision": ASSETS_REVISION,
        "trials": len(episodes), "successes": successes, "success_rate": successes / len(episodes),
        "episodes": episodes,
    }
    write_json(folder / "report.json", report)
    print(f"Results: {folder}", flush=True)
    return folder

"""

## 実行するタスクIDを選ぶ

下の一覧から `TASK_ID` を選んで、このセルから下を実行してください。どのタスクも黒いボウルを皿に移す課題です。IDによって、ボウルが最初に置かれている場所が違います。

| ID | 黒いボウルの場所 |
| ---: | --- |
| 0 | 皿と小鉢の間 |
| 1 | 小鉢の横 |
| 2 | テーブルの中央 |
| 3 | クッキー箱の上 |
| 4 | 木製キャビネットの一番上の引き出しの中 |
| 5 | 小鉢の上 |
| 6 | クッキー箱の横 |
| 7 | コンロの上 |
| 8 | 皿の横 |
| 9 | 木製キャビネットの上 |

初期設定では最大20ステップ動かします。04の学習後の成功率と比べる場合は、IDを `0` にし、セルのコードを開いて `CONNECTION_CHECK` を `False` にします。

In [ ]:
# @title 実行するタスクIDを選ぶ
TASK_ID = "0"  # @param ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
CONNECTION_CHECK = True

config = json.loads((PROJECT / "evaluation.json").read_text())
config["task_ids"] = [int(TASK_ID)]
if CONNECTION_CHECK:
    config["max_steps"] = 20
    config["initial_states"]["validation"] = [0]

print("タスクID:", TASK_ID)
print("開始時の配置:", config["initial_states"]["validation"])
print("1試行のステップ上限:", config["max_steps"])
(PROJECT / "active_evaluation.json").write_text(json.dumps(config))

## LIBEROで推論・評価する

In [ ]:
result = run_python(
    MODEL_CODE
    + EVALUATION_CODE
    + r"""
from common import read_json

evaluate(read_json("active_evaluation.json"), split="validation")
"""
)

## タスクごとの指示と動画を見る

各動画の上に、そのタスクでモデルに渡した指示と結果が出ます。短い試行では、まずボウルがどこにあり、ロボットがどちらに手を伸ばしたかを見てください。

In [ ]:
from IPython.display import Markdown, Video, display

report = json.loads((result / "report.json").read_text())
print(f"成功数 / 試行数: {report['successes']} / {report['trials']}")
for episode in report["episodes"]:
    title = f"タスク {episode['task_id']}（開始時の配置 {episode['initial_state']}）"
    outcome = "成功" if episode["success"] else "未達"
    display(Markdown(
        f"### {title}\n\n"
        f"**指示** {episode['instruction']}\n\n"
        f"**結果** {outcome}（{episode['steps']}ステップ）"
    ))
    display(Video(str(result / episode["video"]), embed=True))

## 動画で見ておくこと

動画でボウルの位置とロボットが手を伸ばした方向を見てください。別のタスクも見たい場合は、たとえばIDを `3` に変え、タスクを選ぶセルからもう一度実行します。

## 結果をDriveに保存する

評価レポートと各初期状態の動画を保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = (
    "02_libero_evaluation_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
)
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)